In [1]:
import time
import csv
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, models
from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights


def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def count_total_params(model):
    return sum(p.numel() for p in model.parameters())


def get_peak_memory():
    if torch.cuda.is_available():
        return torch.cuda.max_memory_allocated() / 1024**2  # MB
    return 0.0

def build_efficientnet_bitfit(num_classes=101):
    model = models.efficientnet_v2_s(weights=EfficientNet_V2_S_Weights.IMAGENET1K_V1)

    # Replace classifier
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)

    # Freeze everything
    for param in model.parameters():
        param.requires_grad = False

    # Unfreeze bias terms
    for name, param in model.named_parameters():
        if "bias" in name:
            param.requires_grad = True

    # Unfreeze classifier
    for param in model.classifier.parameters():
        param.requires_grad = True

    return model

In [2]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss / len(loader), correct / total


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return total_loss / len(loader), correct / total


In [3]:
os.makedirs("results", exist_ok=True)
os.makedirs("models", exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

csv_file = "results/EfficientNetV2S_BitFit_metrics.csv"

if not os.path.exists(csv_file):
    with open(csv_file, mode="w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "method", "epoch",
            "train_loss", "train_acc",
            "val_loss", "val_acc",
            "epoch_time_sec",
            "peak_memory_mb",
            "trainable_params",
            "total_params"
        ])

weights = EfficientNet_V2_S_Weights.IMAGENET1K_V1
transform = weights.transforms()

# Load Food101
train_dataset_full = datasets.Food101(
    root="./data",
    split="train",
    download=True,
    transform=transform
)

test_dataset = datasets.Food101(
    root="./data",
    split="test",
    download=True,
    transform=transform
)

# Train / validation split
train_size = int(0.9 * len(train_dataset_full))
val_size = len(train_dataset_full) - train_size
train_dataset, val_dataset = random_split(train_dataset_full, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=4, pin_memory=True)

# Build model
model = build_efficientnet_bitfit(num_classes=101).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4
)

num_epochs = 8
best_val_acc = 0.0

trainable_params = count_trainable_params(model)
total_params = count_total_params(model)

print(f"Total params: {total_params:,}")
print(f"Trainable params: {trainable_params:,}")
print(f"Trainable %: {100 * trainable_params / total_params:.4f}%")

print("\nTrainable parameter names:")
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name, param.shape)

for epoch in range(num_epochs):
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    start_time = time.time()

    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    epoch_time = time.time() - start_time
    peak_memory = get_peak_memory()

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f}, Val Acc:   {val_acc:.4f}")
    print(f"Time: {epoch_time:.2f}s | Peak Mem: {peak_memory:.2f} MB")
    print("-" * 40)

    # Save metrics
    with open(csv_file, mode="a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "efficientnet_bitfit",
            epoch + 1,
            train_loss,
            train_acc,
            val_loss,
            val_acc,
            epoch_time,
            peak_memory,
            trainable_params,
            total_params
        ])

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc

        torch.save({
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_acc": val_acc,
            "epoch": epoch
        }, "models/EfficientNetV2S_BitFit_best.pth")

        print("Best model saved!")

print(f"\nBest validation accuracy: {best_val_acc:.4f}")

test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")

Using device: cuda
Total params: 20,306,869
Trainable params: 241,509
Trainable %: 1.1893%

Trainable parameter names:
features.0.1.bias torch.Size([24])
features.1.0.block.0.1.bias torch.Size([24])
features.1.1.block.0.1.bias torch.Size([24])
features.2.0.block.0.1.bias torch.Size([96])
features.2.0.block.1.1.bias torch.Size([48])
features.2.1.block.0.1.bias torch.Size([192])
features.2.1.block.1.1.bias torch.Size([48])
features.2.2.block.0.1.bias torch.Size([192])
features.2.2.block.1.1.bias torch.Size([48])
features.2.3.block.0.1.bias torch.Size([192])
features.2.3.block.1.1.bias torch.Size([48])
features.3.0.block.0.1.bias torch.Size([192])
features.3.0.block.1.1.bias torch.Size([64])
features.3.1.block.0.1.bias torch.Size([256])
features.3.1.block.1.1.bias torch.Size([64])
features.3.2.block.0.1.bias torch.Size([256])
features.3.2.block.1.1.bias torch.Size([64])
features.3.3.block.0.1.bias torch.Size([256])
features.3.3.block.1.1.bias torch.Size([64])
features.4.0.block.0.1.bias t

RuntimeError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1.
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
